# Shor's algorithm

Shor's algorithm factors an integer N in a time that grows polynomially with the number of bits of N, which no known classical algorithm achieves. That is why it matters: the security of RSA rests on factoring large numbers being out of reach, and a large enough quantum computer could do it. This notebook factors 15 and 21 with Polypus, first step by step and then with a single call.

## The idea: factoring is finding an order

Pick a base a that shares no factor with N. Its **order** r is the smallest positive integer with a^r ≡ 1 (mod N). If r is even and a^(r/2) is not ≡ −1 (mod N), N divides (a^(r/2) − 1)(a^(r/2) + 1) without dividing either of them, so gcd(a^(r/2) − 1, N) and gcd(a^(r/2) + 1, N) are divisors of N. With N = 15 and a = 7, by hand:

In [ ]:
import math

N, a = 15, 7
print([pow(a, k, N) for k in range(1, 5)])  # 7, 4, 13, 1: the order is r = 4

x = pow(a, 4 // 2, N)  # 7^2 mod 15 = 4
print(math.gcd(x - 1, N), math.gcd(x + 1, N))  # 3 and 5

Trying powers one by one takes on the order of N steps, an amount exponential in the number of bits. The quantum part of the algorithm finds r with a circuit of polynomial size; the rest is classical.

## The circuit

`polypus.circuits.templates.shor_order_finding(a, N)` builds that order-finding circuit, without measurements:

In [ ]:
from polypus.circuits.templates import shor_order_finding

qc = shor_order_finding(7, 15)
print(repr(qc))

That is 18 qubits and 8,561 gates, too many to draw. With n = 4 bits for N = 15, the qubits are laid out like this:

- **0 to 7, the counting register**, of t = 2n = 8 qubits. They start in superposition with H, and qubit k controls a multiplication of the work register by a^(2^k) mod N. At the end, the transform of [`04b`](04b_qft.ipynb), inverted, turns the accumulated phases into a number that can be measured.
- **8 to 11, the work register**, of n qubits. It starts at 1 and every controlled multiplication multiplies it.
- **12 to 16, an auxiliary register** of n + 1 qubits, and **17, an ancilla**. The modular multiplication uses them, and they return to 0.

## Running it

It is enough to measure the counting register, each qubit i into classical bit i, and run the circuit on the native simulator:

In [ ]:
import polypus

for i in range(8):
    qc.measure(i, i)

result = polypus.run_quantum_circuit(
    qc, shots=1024, infrastructure="local", backend="polypus", seed=7
)
counts = result.merged_counts
print(sorted((int(bits, 2), n) for bits, n in counts.items()))

In [ ]:
import matplotlib.pyplot as plt

plt.bar([int(bits, 2) for bits in counts], list(counts.values()), width=6)
plt.xlabel("Measured value y")
plt.ylabel("Number of times")
plt.title("Counting register of shor_order_finding(7, 15)")
plt.show()

Read as an integer, with qubit 0 as the least significant bit, the counting register gives only four values, equally often: 0, 64, 128 and 192. They are the multiples of 2^t / r = 256 / 4 = 64, that is, y = s · 2^t / r with s = 0, 1, 2 and 3. Each measurement approximates a fraction s / r, exactly here because r = 4 divides 2^8.

## From the result to the order

`polypus.shor.order_from_measurement(y, t, a, N)` expands y / 2^t as a continued fraction and tries the denominators of its approximations, and some of their multiples, until it finds an r with a^r ≡ 1 (mod N). Then `factors_from_order(a, N, r)` takes the gcd step:

In [ ]:
from polypus.shor import factors_from_order, order_from_measurement

print(order_from_measurement(64, 8, 7, 15))  # 64/256 = 1/4: r = 4
print(order_from_measurement(128, 8, 7, 15))  # 128/256 = 1/2: tries 2, then 4
print(order_from_measurement(0, 8, 7, 15))  # 0/256 says nothing about r
print(factors_from_order(7, 15, 4))

The peak at 0 is useless: it is s = 0, and 0 / 2^t equals 0 / r for any r. It comes up in one measurement out of r, so it is enough to move on to the next most frequent result. The peak at 128 gives the fraction 1/2, whose denominator divides the order but is not the order: that is why its multiples are tried too.

## All together

`polypus.shor.factor` goes the whole way. It settles without a circuit the cases that need none (an even N, a power of an integer, a base sharing a factor with N); otherwise it picks a random base, runs the circuit, processes the measurements and, if no factors come out, tries again with another base. With `seed` the result is reproducible:

In [ ]:
from polypus.shor import factor

result = factor(15, infrastructure="local", backend="polypus", seed=7)
print(result)
assert result.factors == (3, 5)
assert sorted(int(bits, 2) for bits in result.counts) == [0, 64, 128, 192]

`method="quantum"` says the order was read off the counts of a run of the circuit, and `attempts` counts those runs. It does not prove that the circuit worked: the reading tries every outcome and several multiples of each order, so for the N that can be simulated it finds the order even in counts with no signal. The evidence is in `counts`: here, with a = 7, they are the four peaks from before, 0, 64, 128 and 192. With N = 21 and a = 2, whose order is 6, the counting register can stay at 6 qubits instead of 2n = 10: 18 qubits in all instead of 22. Since 6 does not divide 2^6 = 64, the peaks are no longer exact and spread over the values near 64 · s / 6, but the continued fraction still finds the order:

In [ ]:
result = factor(
    21, a=2, counting_qubits=6, infrastructure="local", backend="polypus", seed=1
)
print(result.factors, result.method, result.order)
assert result.factors == (3, 7)

## How far it goes

The circuit has t + 2n + 2 qubits; with the default counting register, t = 2n, that is 4n + 2. The local simulators hold up to 30 qubits, enough for N < 128, of up to 7 bits, but each qubit more doubles the time and memory of a run: on four cores an attempt takes seconds for N < 16, about a minute for N < 32, half an hour for N < 64 (26 qubits, 1 GiB) and, at 30 qubits, 16 GiB and many hours. The template builds circuits of up to 24 bits, useful to estimate resources even when they cannot be simulated. For example, for a 16-bit N:

In [ ]:
big = shor_order_finding(2, 251 * 257)  # N = 64507, 16 bits
print(repr(big))

That is 66 qubits and about 849,000 gates. `to_qasm3` exports these circuits up to 16 bits; from 17 on, the text would exceed the 32 MiB the OpenQASM 3 importer reads. A smaller `counting_qubits` saves qubits and makes each run cheaper, in exchange for less precise counts.

## Summary

This notebook factored 15 and 21 with Shor's algorithm: finding the order of a modulo N with `shor_order_finding`, reading the order with continued fractions with `order_from_measurement`, the gcds that give the factors with `factors_from_order`, and all of it together with `polypus.shor.factor`.

The next section covers the oracle search algorithms: Deutsch-Jozsa and Grover.

## Next step

Continue with: [`05a_deutsch_jozsa.ipynb`](05a_deutsch_jozsa.ipynb).